# multi_strat_003

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (116).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 11 |
| Detected functions | calculate_indicators, backtest_strategy_for_ticker, backtest_strategy_for_all_tickers, backtest_strategy, weighted_moving_average, calculate_tmf, apply_tmf_to_tickers, backtest_strategy, plot_trade_trace, plot_trades_with_hover_labels, weighted_moving_average, calculate_tmf |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/filtered_data.csv')
data

In [ ]:
data = data[data['Ticker'] == 'TATASTEEL']
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm  # For progress bar

# Function to calculate indicators in a vectorized way
def calculate_indicators(data):
    # Twiggs Money Flow
    hlc = (data['close'] - data['low'] - (data['high'] - data['close']))
    data['TMF'] = (hlc * data['volume']).rolling(14).sum() / data['volume'].rolling(14).sum()

    # VWAP
    data['VWAP'] = (data['close'] * data['volume']).cumsum() / data['volume'].cumsum()

    # Kalman Filter (approximated with EMA for simplicity)
    data['Kalman'] = data['close'].ewm(span=10).mean()

    # ADX
    high_low = data['high'] - data['low']
    high_close = abs(data['high'] - data['close'].shift(1))
    low_close = abs(data['low'] - data['close'].shift(1))
    tr = pd.Series(np.maximum(high_low, np.maximum(high_close, low_close)))

    pdm = pd.Series(np.where(data['high'] > data['high'].shift(1), data['high'] - data['high'].shift(1), 0))
    ndm = pd.Series(np.where(data['low'].shift(1) > data['low'], data['low'].shift(1) - data['low'], 0))

    atr = tr.rolling(14).mean()
    pdm_smoothed = pdm.rolling(14).mean()
    ndm_smoothed = ndm.rolling(14).mean()

    di_plus = 100 * (pdm_smoothed / atr)
    di_minus = 100 * (ndm_smoothed / atr)
    dx = 100 * abs(di_plus - di_minus) / (di_plus + di_minus)

    data['ADX'] = dx.rolling(14).mean()

    return data


# Optimized backtesting function for a single ticker
def backtest_strategy_for_ticker(data):
    # Pre-compute indicators
    data = calculate_indicators(data)

    # Add a column to mark entry/exit signals
    data['Signal'] = 0
    data['Signal'] = np.where(
        (data['Kalman'] > data['Kalman'].shift(1)) &
        (data['TMF'] > 0) &
        (data['close'] > data['VWAP']) &
        (data['ADX'] > 20),
        1,
        np.where(
            (data['Kalman'] < data['Kalman'].shift(1)) &
            (data['TMF'] < 0) &
            (data['close'] < data['VWAP']) &
            (data['ADX'] > 20),
            -1,
            0
        )
    )

    # Add time filtering for trades (e.g., exclude after 15:25)
    data['time'] = pd.to_datetime(data['date']).dt.time
    data['Signal'] = np.where(data['time'] > pd.to_datetime("15:25:00").time(), 0, data['Signal'])

    # Initialize trade variables
    tradebook = []
    position = 0
    entry_price = 0
    cumulative_pnl = 0
    capital = 100000  # Fixed capital per trade
    brokerage = 0.001  # 0.1% brokerage

    # Loop through rows to generate trades
    for i, row in data.iterrows():
        # Closing open positions at 15:28
        if position != 0 and row['time'] == pd.to_datetime("15:28:00").time():
            exit_price = row['close']
            pnl = (exit_price - entry_price) * position * capital / entry_price
            pnl -= abs(pnl) * brokerage
            cumulative_pnl += pnl
            tradebook.append({
                'Date Entry': entry_date,
                'Date Exit': row['date'],
                'Entry Price': entry_price,
                'Exit Price': exit_price,
                'PnL': pnl,
                'Action': 'Close',
                'Cumulative PnL': cumulative_pnl
            })
            position = 0
            continue

        # Entry signal
        if position == 0 and row['Signal'] != 0:
            position = row['Signal']
            entry_price = row['close']
            entry_date = row['date']
            tradebook.append({
                'Date Entry': row['date'],
                'Date Exit': None,
                'Entry Price': entry_price,
                'Exit Price': None,
                'PnL': None,
                'Action': 'Buy' if position > 0 else 'Sell',
                'Cumulative PnL': cumulative_pnl
            })

        # Exit signal
        elif position != 0 and ((position > 0 and row['close'] < row['VWAP']) or (position < 0 and row['close'] > row['VWAP'])):
            exit_price = row['close']
            pnl = (exit_price - entry_price) * position * capital / entry_price
            pnl -= abs(pnl) * brokerage
            cumulative_pnl += pnl
            tradebook.append({
                'Date Entry': entry_date,
                'Date Exit': row['date'],
                'Entry Price': entry_price,
                'Exit Price': exit_price,
                'PnL': pnl,
                'Action': 'Close',
                'Cumulative PnL': cumulative_pnl
            })
            position = 0

    # Convert tradebook to DataFrame
    tradebook_df = pd.DataFrame(tradebook)
    return tradebook_df

# Optimized backtesting for multiple tickers with a progress bar
def backtest_strategy_for_all_tickers(data):
    all_tradebooks = []
    tickers = data['Ticker'].unique()

    # Use tqdm for progress bar
    for ticker in tqdm(tickers, desc="Processing tickers"):
        ticker_data = data[data['Ticker'] == ticker].copy()
        ticker_data.sort_values(by='date', inplace=True)  # Ensure data is sorted
        tradebook = backtest_strategy_for_ticker(ticker_data)
        tradebook['Ticker'] = ticker  # Add ticker column to tradebook
        all_tradebooks.append(tradebook)

    # Combine all tradebooks into a single DataFrame
    combined_tradebook = pd.concat(all_tradebooks, ignore_index=True)
    return combined_tradebook

# Example usage
# data = pd.read_csv("your_data.csv")  # Load your data with all tickers
combined_tradebook = backtest_strategy_for_all_tickers(data)
combined_tradebook.to_csv("tradebook.csv", index=False)  # Save the tradebook


In [ ]:
t = combined_tradebook.to_csv("/content/drive/MyDrive/tradebook.csv", index=False)

In [ ]:
import plotly.graph_objects as go


# Plotting with Plotly
fig = go.Figure()

# Add Close Price
fig.add_trace(go.Scatter(
    x=data['date'],
    y=data['close'],
    mode='lines',
    name='Close Price',
    line=dict(color='blue'),
    hoverinfo='x+y'
))

# Add TMF
fig.add_trace(go.Scatter(
    x=data['date'],
    y=data['TMF'],
    mode='lines',
    name='TMF',
    line=dict(color='aqua'),
    hoverinfo='x+y'
))

# Add Positive TMF (Green Area)
fig.add_trace(go.Scatter(
    x=data['date'],
    y=data['TMF_Positive'],
    mode='lines',
    name='Positive TMF (>0.2499)',
    line=dict(color='green'),
    fill='tozeroy',  # Fills the area under the line
    hoverinfo='x+y'
))

# Add Negative TMF (Red Area)
fig.add_trace(go.Scatter(
    x=data['date'],
    y=data['TMF_Negative'],
    mode='lines',
    name='Negative TMF (<-0.2499)',
    line=dict(color='red'),
    fill='tozeroy',  # Fills the area under the line
    hoverinfo='x+y'
))

# Update layout for shared hover
fig.update_layout(
    title="Close Price and Twiggs Money Flow (TMF)",
    xaxis_title="Date",
    yaxis_title="Value",
    hovermode='x unified',  # Ensures both traces are shown on hover
    legend=dict(orientation="h", yanchor="bottom", y=1, xanchor="right", x=1)
)

# Show the plot
fig.show()

In [ ]:
import pandas as pd

# Assuming 'result' is the DataFrame after calculating TMF
result['date'] = pd.to_datetime(result['date'])

# Placeholder for the tradebook to store the results
tradebook = []

# Brokerage constant (apply to both entry and exit)
BROKERAGE = 0.001

# Capital per trade (Assumed to be a fixed amount for each trade)
TRADE_CAPITAL = 100000

# Take profit and stop loss percentages
TAKE_PROFIT = 0.02  # 2%
STOP_LOSS = 0.01    # 1%

# Function to backtest the strategy
def backtest_strategy(data):
    global tradebook

    # Iterate through each ticker
    for ticker in data['Ticker'].unique():
        ticker_data = data[data['Ticker'] == ticker].copy()
        ticker_data.sort_values(by='date', inplace=True)

        # Initialize variables
        position = None
        entry_price = 0
        entry_time = None
        cum_pnl = 0
        quantity = 0

        # Iterate through rows
        for i, row in ticker_data.iterrows():
            current_time = row['date'].time()

            if position is None:  # No active position
                if row['TMF'] > 0.2499:  # Signal to go long
                    position = 'long'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price  # Calculate the number of shares/contracts bought
                elif row['TMF'] < -0.2499:  # Signal to go short
                    position = 'short'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price  # Calculate the number of shares/contracts sold

            # Manage open positions
            if position is not None:
                exit_reason = None
                exit_price = 0

                if position == 'long':
                    # Check take profit and stop loss
                    tp_price = entry_price * (1 + TAKE_PROFIT)
                    sl_price = entry_price * (1 - STOP_LOSS)
                    if row['high'] >= tp_price:
                        exit_price = tp_price
                        exit_reason = 'Take Profit'
                    elif row['low'] <= sl_price:
                        exit_price = sl_price
                        exit_reason = 'Stop Loss'
                elif position == 'short':
                    # Check take profit and stop loss
                    tp_price = entry_price * (1 - TAKE_PROFIT)
                    sl_price = entry_price * (1 + STOP_LOSS)
                    if row['low'] <= tp_price:
                        exit_price = tp_price
                        exit_reason = 'Take Profit'
                    elif row['high'] >= sl_price:
                        exit_price = sl_price
                        exit_reason = 'Stop Loss'

                # Close all trades at 15:28
                if current_time.hour == 15 and current_time.minute == 28 and exit_reason is None:
                    exit_price = row['close']
                    exit_reason = 'End of Day'

                # Record the trade if exited
                if exit_reason is not None:
                    exit_time = row['date']

                    # Calculate PnL, including brokerage fees for both entry and exit
                    if position == 'long':
                        trade_cost = entry_price * quantity * BROKERAGE
                        exit_cost = exit_price * quantity * BROKERAGE
                        pnl = ((exit_price - entry_price) * quantity) - (trade_cost + exit_cost)
                    elif position == 'short':
                        trade_cost = entry_price * quantity * BROKERAGE
                        exit_cost = exit_price * quantity * BROKERAGE
                        pnl = ((entry_price - exit_price) * quantity) - (trade_cost + exit_cost)

                    cum_pnl += pnl

                    tradebook.append({
                        'Ticker': ticker,
                        'Entry Time': entry_time,
                        'Exit Time': exit_time,
                        'Entry Price': entry_price,
                        'Exit Price': exit_price,
                        'PnL': pnl,
                        'Exit Reason': exit_reason,
                        'Cumulative PnL': cum_pnl
                    })

                    # Reset position
                    position = None
                    entry_price = 0
                    entry_time = None
                    quantity = 0

        # At the end of the day, make sure no open positions are left
        if position is not None:
            exit_price = ticker_data.iloc[-1]['close']
            exit_time = ticker_data.iloc[-1]['date']

            if position == 'long':
                trade_cost = entry_price * quantity * BROKERAGE
                exit_cost = exit_price * quantity * BROKERAGE
                pnl = ((exit_price - entry_price) * quantity) - (trade_cost + exit_cost)
            elif position == 'short':
                trade_cost = entry_price * quantity * BROKERAGE
                exit_cost = exit_price * quantity * BROKERAGE
                pnl = ((entry_price - exit_price) * quantity) - (trade_cost + exit_cost)

            cum_pnl += pnl

            tradebook.append({
                'Ticker': ticker,
                'Entry Time': entry_time,
                'Exit Time': exit_time,
                'Entry Price': entry_price,
                'Exit Price': exit_price,
                'PnL': pnl,
                'Exit Reason': 'End of Day',
                'Cumulative PnL': cum_pnl
            })

# Example usage
# Assuming 'result' is the DataFrame with the TMF values
backtest_strategy(result)

# Convert tradebook to DataFrame for easier analysis
tradebook_df = pd.DataFrame(tradebook)
print(tradebook_df)

# Save the tradebook to a CSV file
tradebook_df.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np

# Vectorized Weighted Moving Average (WiMA)
def weighted_moving_average(series, length):
    weights = np.arange(1, length + 1)  # Create a weight vector
    wma = np.convolve(series, weights[::-1], mode='valid') / weights.sum()  # Use convolution for efficiency
    return np.concatenate([np.full(length - 1, np.nan), wma])

def calculate_tmf(data, length=21):
    """
    Calculate Twiggs Money Flow (TMF) indicator efficiently.
    """
    close = data['close']
    high = data['high']
    low = data['low']
    volume = data['volume']

    # Calculate True High, True Low, and True Range (vectorized)
    tr_h = np.maximum(close.shift(1), high)
    tr_l = np.minimum(close.shift(1), low)
    tr_c = tr_h - tr_l

    # Adjusted Volume (adv), vectorized
    adv = volume * ((close - tr_l) - (tr_h - close)) / np.where(tr_c == 0, np.nan, tr_c)
    adv.fillna(0, inplace=True)

    # Calculate WiMA (vectorized)
    wmV = weighted_moving_average(volume.values, length)
    wmA = weighted_moving_average(adv.values, length)

    # Calculate TMF, Positives, and Negatives (vectorized)
    tmf = np.where(wmV == 0, 0, wmA / wmV)

    # Add results to DataFrame
    data['TMF'] = tmf

    return data

def apply_tmf_to_tickers(data, length):
    """
    Apply TMF calculation to each ticker efficiently.
    """
    data = data.sort_values(by=['Ticker', 'date'])

    # Apply TMF to all data in one go
    return data.groupby('Ticker').apply(calculate_tmf, length=length).reset_index(drop=True)

def backtest_strategy(data):
    """
    Perform backtesting using TMF-based strategy.

    Args:
        data (DataFrame): Input data with TMF values.
    """
    tradebook = []
    BROKERAGE = 0.001
    TRADE_CAPITAL = 100000
    TAKE_PROFIT = 0.03
    STOP_LOSS = 0.015

    # Initialize cumulative PnL
    cum_pnl = 0
    data['date'] = pd.to_datetime(data['date'])

    # Precompute take-profit and stop-loss levels
    data['tp_price_long'] = data['close'] * (1 + TAKE_PROFIT)
    data['sl_price_long'] = data['close'] * (1 - STOP_LOSS)
    data['tp_price_short'] = data['close'] * (1 - TAKE_PROFIT)
    data['sl_price_short'] = data['close'] * (1 + STOP_LOSS)

    # Iterate through each Ticker
    for ticker in data['Ticker'].unique():
        ticker_data = data[data['Ticker'] == ticker].copy().sort_values(by='date')

        # Initialize position variables
        position, entry_price, entry_time, quantity = None, 0, None, 0

        # Iterate through the data
        for i, row in ticker_data.iterrows():
            current_time = row['date'].time()

            # Skip trades on Thursdays
            if row['date'].weekday() in (0, 3):  # Skip Monday (0) and Thursday (3)
              continue

            # Entry signals
            if position is None and current_time <= pd.Timestamp("15:25").time():  # Restrict trades after 15:25
                if row['TMF'] > 0.2499:
                    position = 'long'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price
                elif row['TMF'] < -0.2499:
                    position = 'short'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price

            # Exit signals
            if position is not None:
                exit_reason, exit_price = None, 0

                if position == 'long':
                    if row['high'] >= row['tp_price_long']:
                        exit_price = row['tp_price_long']
                        exit_reason = 'Take Profit'
                    elif row['low'] <= row['sl_price_long']:
                        exit_price = row['sl_price_long']
                        exit_reason = 'Stop Loss'
                elif position == 'short':
                    if row['low'] <= row['tp_price_short']:
                        exit_price = row['tp_price_short']
                        exit_reason = 'Take Profit'
                    elif row['high'] >= row['sl_price_short']:
                        exit_price = row['sl_price_short']
                        exit_reason = 'Stop Loss'

                # Close all trades at 15:28
                if current_time.hour == 15 and current_time.minute == 28 and exit_reason is None:
                    exit_price = row['close']
                    exit_reason = 'End of Day'

                # Record the trade
                if exit_reason:
                    exit_time = row['date']
                    trade_cost = entry_price * quantity * BROKERAGE
                    exit_cost = exit_price * quantity * BROKERAGE
                    pnl = ((exit_price - entry_price) * quantity if position == 'long' else
                           (entry_price - exit_price) * quantity) - (trade_cost + exit_cost)

                    cum_pnl += pnl

                    tradebook.append({
                        'Ticker': ticker,
                        'Entry Time': entry_time,
                        'Exit Time': exit_time,
                        'Entry Price': entry_price,
                        'Exit Price': exit_price,
                        'PnL': pnl,
                        'Exit Reason': exit_reason,
                        'Cumulative PnL': cum_pnl,
                        'DateTime': entry_time  # Add DateTime column
                    })

                    # Reset position
                    position, entry_price, entry_time, quantity = None, 0, None, 0

    # Convert tradebook to DataFrame
    tradebook_df = pd.DataFrame(tradebook)

    # Save the tradebook to a CSV file
    tradebook_df.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)
    print(tradebook_df)

# Example usage
length = 17
data_with_tmf = apply_tmf_to_tickers(data, length)
backtest_strategy(data_with_tmf)


In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plot_trade_trace(entry_time, exit_time, entry_price, exit_price, pnl, exit_reason, tp_level, sl_level, trade_type):
    """
    Helper function to create trade traces for entry, exit, connecting lines, TP, and SL.
    """
    traces = []

    # Determine colors based on trade type and PnL
    entry_color = 'green' if trade_type == 'long' else 'orange'
    exit_color = 'blue' if trade_type == 'long' else 'red'
    line_color = 'darkgreen' if pnl > 0 else 'darkred'
    entry_label = "Long Entry" if trade_type == 'long' else "Short Entry"
    exit_label = "Long Exit" if trade_type == 'long' else "Short Exit"

    # Entry scatter point
    traces.append(go.Scatter(
        x=[entry_time],
        y=[entry_price],
        mode='markers',
        marker=dict(color=entry_color, size=10, symbol="triangle-up"),
        name=entry_label,
        hovertemplate=f"<b>{entry_label}</b><br>Price: {entry_price:.2f}<br>Time: {entry_time}<extra></extra>"
    ))

    # Exit scatter point
    traces.append(go.Scatter(
        x=[exit_time],
        y=[exit_price],
        mode='markers',
        marker=dict(color=exit_color, size=10, symbol="triangle-down"),
        name=exit_label,
        hovertemplate=f"<b>{exit_label}</b><br>Price: {exit_price:.2f}<br>Time: {exit_time}<br>PnL: {pnl:.2f}<extra></extra>"
    ))

    # Connecting line between entry and exit
    traces.append(go.Scatter(
        x=[entry_time, exit_time],
        y=[entry_price, exit_price],
        mode='lines',
        line=dict(color=line_color, width=2),
        name="PnL Positive" if pnl > 0 else "PnL Negative",
        hovertemplate=f"<b>Trade Line</b><br>Entry: {entry_price:.2f}<br>Exit: {exit_price:.2f}<br>PnL: {pnl:.2f}<extra></extra>"
    ))

    # Plot Take Profit and Stop Loss lines if defined
    if pd.notna(tp_level):
        traces.append(go.Scatter(
            x=[entry_time, exit_time],
            y=[tp_level, tp_level],
            mode='lines',
            line=dict(color='green', dash='dot'),
            name="Take Profit",
            hovertemplate=f"<b>Take Profit</b><br>Level: {tp_level:.2f}<extra></extra>"
        ))

    if pd.notna(sl_level):
        traces.append(go.Scatter(
            x=[entry_time, exit_time],
            y=[sl_level, sl_level],
            mode='lines',
            line=dict(color='red', dash='dot'),
            name="Stop Loss",
            hovertemplate=f"<b>Stop Loss</b><br>Level: {sl_level:.2f}<extra></extra>"
        ))

    return traces

def plot_trades_with_hover_labels(data, tradebook_df, ticker):
    """
    Plot trades with distinct hover labels for long/short entries and exits,
    and include lines for Take Profit (TP) and Stop Loss (SL) levels until the trade is exited.

    Parameters:
    - data: DataFrame containing price data.
    - tradebook_df: DataFrame containing trade records.
    - ticker: Ticker symbol to filter and plot.
    """
    # Filter data and tradebook for the specific ticker
    ticker_data = data[data['Ticker'] == ticker].sort_values(by='date')
    ticker_trades = tradebook_df[tradebook_df['Ticker'] == ticker]

    # Pre-calculate TP and SL levels for all rows in the data
    ticker_data['tp_price_long'] = ticker_data['close'] * (1 + 0.015)
    ticker_data['sl_price_long'] = ticker_data['close'] * (1 - 0.015)
    ticker_data['tp_price_short'] = ticker_data['close'] * (1 - 0.015)
    ticker_data['sl_price_short'] = ticker_data['close'] * (1 + 0.015)

    # Initialize the figure
    fig = go.Figure()

    # Add close price line
    fig.add_trace(go.Scatter(
        x=ticker_data['date'],
        y=ticker_data['close'],
        mode='lines',
        name='Close Price',
        line=dict(color='blue', width=2),
    ))

    # List to store trade traces
    all_traces = []

    # Add buy/sell signals and connecting lines for each trade
    for _, trade in ticker_trades.iterrows():
        entry_time = trade['Entry Time']
        exit_time = trade['Exit Time']
        entry_price = trade['Entry Price']
        exit_price = trade['Exit Price']
        pnl = trade['PnL']
        exit_reason = trade['Exit Reason']

        # Check and assign TP and SL levels based on trade direction
        if 'long' in exit_reason.lower():
            tp_level = ticker_data.loc[ticker_data['date'] == entry_time, 'tp_price_long'].values[0]
            sl_level = ticker_data.loc[ticker_data['date'] == entry_time, 'sl_price_long'].values[0]
            trade_type = 'long'
        elif 'short' in exit_reason.lower():
            tp_level = ticker_data.loc[ticker_data['date'] == entry_time, 'tp_price_short'].values[0]
            sl_level = ticker_data.loc[ticker_data['date'] == entry_time, 'sl_price_short'].values[0]
            trade_type = 'short'
        else:
            continue  # Skip if the exit reason is not long or short

        # Get trade traces
        trade_traces = plot_trade_trace(entry_time, exit_time, entry_price, exit_price, pnl, exit_reason, tp_level, sl_level, trade_type)

        # Append to the list
        all_traces.extend(trade_traces)

    # Add all trade traces to the figure
    fig.add_traces(all_traces)

    # Update layout for clarity
    fig.update_layout(
        title=f"Trade Visualization for {ticker}",
        xaxis_title="Date",
        yaxis_title="Price",
        legend_title="Trade Actions",
        template="plotly_white",
        hovermode="x unified"
    )

    # Display the plot
    fig.show()

# Example usage:
# Assuming the tradebook from backtest_strategy has been saved in `tradebook.csv`
tradebook_df = pd.read_csv('/content/drive/MyDrive/tradebook.csv')

# Assuming the original data and TMF calculations are available in `data_with_tmf`
plot_trades_with_hover_labels(data_with_tmf, tradebook_df, ticker='RELIANCE')


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm  # For progress bar


# Vectorized Weighted Moving Average (WiMA)
def weighted_moving_average(series, length):
    """
    Calculate the weighted moving average (WiMA) of a given series.

    Args:
        series (pd.Series): The data series.
        length (int): The length of the moving average window.

    Returns:
        np.ndarray: The weighted moving average values.
    """
    weights = np.arange(1, length + 1)  # Create a weight vector
    wma = np.convolve(series, weights[::-1], mode='valid') / weights.sum()  # Use convolution for efficiency
    return np.concatenate([np.full(length - 1, np.nan), wma])  # Prepend NaN values


def calculate_tmf(data, length=21):
    """
    Calculate the TMF (True Market Flow) indicator and add it to the dataframe.

    Args:
        data (pd.DataFrame): The input dataframe with stock data.
        length (int): The length of the moving average window.

    Returns:
        pd.DataFrame: The dataframe with TMF added.
    """
    close, high, low, volume = data['close'], data['high'], data['low'], data['volume']

    # Calculate True High, True Low, and True Range (vectorized)
    tr_h = np.maximum(close.shift(1), high)
    tr_l = np.minimum(close.shift(1), low)
    tr_c = tr_h - tr_l

    # Adjusted Volume (adv), vectorized
    adv = volume * ((close - tr_l) - (tr_h - close)) / np.where(tr_c == 0, np.nan, tr_c)
    adv.fillna(0, inplace=True)

    # Calculate WiMA (vectorized)
    wmV = weighted_moving_average(volume.values, length)
    wmA = weighted_moving_average(adv.values, length)

    # Calculate TMF (vectorized)
    tmf = np.where(wmV == 0, 0, wmA / wmV)

    # Add TMF to DataFrame
    data['TMF'] = tmf

    return data


def apply_tmf_to_tickers(data, length):
    """
    Apply the TMF calculation to each ticker in the data.

    Args:
        data (pd.DataFrame): The input dataframe with stock data.
        length (int): The length of the moving average window.

    Returns:
        pd.DataFrame: The dataframe with TMF for each ticker.
    """
    return data.groupby('Ticker').apply(calculate_tmf, length=length)


def backtest_strategy(data, tmf_positive_threshold, tmf_negative_threshold, take_profit, stop_loss, length):
    """
    Backtest the strategy based on TMF and trade parameters.

    Args:
        data (pd.DataFrame): The input dataframe with stock data and TMF.
        tmf_positive_threshold (float): The threshold for entering long positions.
        tmf_negative_threshold (float): The threshold for entering short positions.
        take_profit (float): The take profit percentage.
        stop_loss (float): The stop loss percentage.
        length (int): The length of the moving average window for TMF.

    Returns:
        float: The cumulative profit and loss (PnL) from the backtest.
        float: The win rate from the backtest.
    """
    tradebook = []
    BROKERAGE = 0.001
    TRADE_CAPITAL = 100000
    cum_pnl = 0
    total_trades = 0
    winning_trades = 0
    data['date'] = pd.to_datetime(data['date'])

    # Backtest for each unique ticker
    for ticker in tqdm(data['Ticker'].unique(), desc="Backtesting Progress"):
        ticker_data = data[data['Ticker'] == ticker].sort_values(by='date')

        position = None
        entry_price = 0
        entry_time = None
        quantity = 0

        # Set price targets for take profit and stop loss
        ticker_data['tp_price_long'] = ticker_data['close'] * (1 + take_profit)
        ticker_data['sl_price_long'] = ticker_data['close'] * (1 - stop_loss)
        ticker_data['tp_price_short'] = ticker_data['close'] * (1 - take_profit)
        ticker_data['sl_price_short'] = ticker_data['close'] * (1 + stop_loss)


        for i, row in ticker_data.iterrows():
            current_time = row['date'].time()

            # Skip trades on Thursdays
            if row['date'].weekday() == 3:  # 3 represents Thursday
                continue

            # Entry signals based on adjustable TMF thresholds
            if position is None and current_time <= pd.Timestamp("15:25").time():
                if row['TMF'] > tmf_positive_threshold:
                    position = 'long'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price
                elif row['TMF'] < tmf_negative_threshold:
                    position = 'short'
                    entry_price = row['close']
                    entry_time = row['date']
                    quantity = TRADE_CAPITAL / entry_price

            # Exit signals
            if position is not None:
                exit_reason = None
                exit_price = 0

                if position == 'long':
                    if row['high'] >= row['tp_price_long']:
                        exit_price = row['tp_price_long']
                        exit_reason = 'Take Profit'
                    elif row['low'] <= row['sl_price_long']:
                        exit_price = row['sl_price_long']
                        exit_reason = 'Stop Loss'
                elif position == 'short':
                    if row['low'] <= row['tp_price_short']:
                        exit_price = row['tp_price_short']
                        exit_reason = 'Take Profit'
                    elif row['high'] >= row['sl_price_short']:
                        exit_price = row['sl_price_short']
                        exit_reason = 'Stop Loss'

                # Close all trades at 15:28
                if current_time.hour == 15 and current_time.minute == 28 and exit_reason is None:
                    exit_price = row['close']
                    exit_reason = 'End of Day'

                if exit_reason is not None:
                    exit_time = row['date']
                    trade_cost = entry_price * quantity * BROKERAGE
                    exit_cost = exit_price * quantity * BROKERAGE
                    if position == 'long':
                        pnl = ((exit_price - entry_price) * quantity) - (trade_cost + exit_cost)
                    elif position == 'short':
                        pnl = ((entry_price - exit_price) * quantity) - (trade_cost + exit_cost)

                    cum_pnl += pnl
                    total_trades += 1
                    if pnl > 0:
                        winning_trades += 1

                    # Log the trade details
                    tradebook.append({
                        'Ticker': ticker,
                        'Entry Time': entry_time,
                        'Exit Time': exit_time,
                        'Entry Price': entry_price,
                        'Exit Price': exit_price,
                        'PnL': pnl,
                        'Exit Reason': exit_reason,
                        'Cumulative PnL': cum_pnl
                    })

                    # Reset position
                    position = None
                    entry_price = 0
                    entry_time = None
                    quantity = 0

    # Calculate win rate
    win_rate = winning_trades / total_trades if total_trades > 0 else 0

    # Convert tradebook to DataFrame and save it
    tradebook_df = pd.DataFrame(tradebook)
    print(tradebook_df)
    tradebook_df.to_csv('tradebook.csv', index=False)

    return cum_pnl, win_rate  # Return the cumulative PnL and win rate


def robustness_test(data, lengths, take_profits, stop_losses, tmf_positive_thresholds, tmf_negative_thresholds):
    """
    Perform robustness testing on the strategy by varying the parameters.

    Args:
        data (pd.DataFrame): The input dataframe with stock data.
        lengths (list): The list of lengths for TMF calculation.
        take_profits (list): The list of take profit percentages.
        stop_losses (list): The list of stop loss percentages.
        tmf_positive_thresholds (list): The list of positive TMF thresholds.
        tmf_negative_thresholds (list): The list of negative TMF thresholds.

    Returns:
        pd.DataFrame: The results of the robustness test.
    """
    results = []

    # Precompute TMF for all lengths for efficiency
    for length in lengths:
        data_with_tmf = apply_tmf_to_tickers(data.copy(), length=length)

        for tp in take_profits:
            for sl in stop_losses:
                for tmf_pos in tmf_positive_thresholds:
                    for tmf_neg in tmf_negative_thresholds:
                        print(f"Testing Length={length}, TP={tp}, SL={sl}, TMF_Positive={tmf_pos}, TMF_Negative={tmf_neg}")

                        # Get cumulative PnL and win rate for this parameter set
                        cum_pnl, win_rate = backtest_strategy(data_with_tmf,
                                                              tmf_positive_threshold=tmf_pos,
                                                              tmf_negative_threshold=tmf_neg,
                                                              take_profit=tp,
                                                              stop_loss=sl,
                                                              length=length)

                        print(f"Cumulative PnL for Length={length}, TP={tp}, SL={sl}, TMF_Positive={tmf_pos}, TMF_Negative={tmf_neg}: {cum_pnl}")
                        print(f"Win Rate: {win_rate:.2f}")

                        # Store the results with the cumulative PnL and win rate
                        results.append({
                            'Length': length,
                            'Take Profit': tp,
                            'Stop Loss': sl,
                            'TMF_Positive Threshold': tmf_pos,
                            'TMF_Negative Threshold': tmf_neg,
                            'Cumulative PnL': cum_pnl,
                            'Win Rate': win_rate
                        })

    return pd.DataFrame(results)


# Example usage
lengths = [9, 12, 14, 15, 17, 21, 31, 41, 51]
take_profits = [0.015, 0.02, 0.03]
stop_losses = [0.015, 0.02]
tmf_positive_thresholds = [0.25]
tmf_negative_thresholds = [-0.25]

results_df = robustness_test(data, lengths=lengths, take_profits=take_profits, stop_losses=stop_losses,
                             tmf_positive_thresholds=tmf_positive_thresholds, tmf_negative_thresholds=tmf_negative_thresholds)

print(results_df)


In [ ]:
t = results_df.to_csv('result')